# Scheduling against uncertain electricity demand

Choose capacity and dispatch to meet a required probability of serving demand. Under a normal distribution for demand in each block, the chance constraint is a deterministic lower bound on scheduled delivery. This port uses the normal quantile directly.

For block $t$, if $D_t \sim \mathcal{N}(\mu_t,\sigma_t^2)$, then $\Pr(D_t \le q_t) \ge p$ is equivalent to $q_t \ge \mu_t + \sigma_t \Phi^{-1}(p)$. The probability applies to each block separately.

## Estimate a demand distribution

The means and standard deviations come from ERCOT's COAST hourly load data in July 2019, using the reported Houston local hours. First average each day's load within six-hour blocks, then compute each block's mean and sample standard deviation across days. Scale both statistics by the same factor to represent a local system.

In [1]:
from statistics import NormalDist

import pandas as pd
from energia import Model, Periods, Process, Resource
from gana import inf

mean_load = [12438.75, 14181.52, 18082.15, 16108.82]
std_load = [697.47, 876.95, 1147.07, 915.28]
scale = 40 / max(mean_load)
distributions = [
    NormalDist(mean * scale, sd * scale)
    for mean, sd in zip(mean_load, std_load, strict=True)
]
solar = [0.04, 0.55, 0.62, 0.07]
wind = [0.42, 0.31, 0.25, 0.5]

## Compare planning targets

A higher service probability requires a larger delivery in each block. Renewable dispatch is limited by weather, while a gas generator can cover firm demand.

In [2]:
def schedule(probability):
    m = Model(f"service_{probability}")
    m.block = Periods()
    m.day = len(distributions) * m.block
    m.declare(Resource, ["sun", "wind", "gas", "power"])
    for resource in (m.sun, m.wind, m.gas):
        _ = resource.consume(m.block) >= 0

    served = m.power.release(m.block)
    _ = served >= [distribution.inv_cdf(probability) for distribution in distributions]
    cost = 0
    for name, conversion, availability, capex, opex in (
        ("pv", m.power == -m.sun, solar, 90, 0),
        ("wind_farm", m.power == -m.wind, wind, 110, 0),
        ("gas_plant", m.power == -0.108 * m.gas, 1, 80, 50),
    ):
        plant = Process(conversion)
        setattr(m, name, plant)
        capacity = plant.capacity(m.day)
        dispatch = plant.operate(m.block)
        _ = capacity <= 100
        _ = dispatch <= availability
        plant.locate()
        cost += capex * capacity.V()[0] + 6 * opex * sum(dispatch.V())

    m.program.daily_cost = inf(cost)
    m.opt()
    achieved = [
        distribution.cdf(delivery)
        for distribution, delivery in zip(distributions, served.output(aslist=True), strict=True)
    ]
    assert min(achieved) >= probability - 1e-6
    return {
        "Daily cost": m.program.active_objective.X,
        "Minimum block service probability": min(achieved),
        "Firm capacity (MW)": m.gas_plant.capacity(m.day).output(aslist=True)[0],
    }


comparison = pd.DataFrame({target: schedule(target) for target in [0.5, 0.9, 0.95, 0.99]}).T
comparison.index.name = "Service target"
assert comparison["Daily cost"].is_monotonic_increasing
comparison.round(4)

,Daily cost,Minimum block service probability,Firm capacity (MW)
Service target,,,
0.50,10668.4029,0.50,0.0
0.90,11488.4945,0.90,0.0
0.95,11720.9791,0.95,0.0
0.99,12157.0816,0.99,0.0
